# 01 Build Case Base

Notebook ini digunakan untuk tahap pertama sistem Case-Based Reasoning (CBR), yaitu membangun case base dari 30 dokumen putusan pengadilan.

Dataset yang digunakan:
- Jenis perkara: Pidana Umum – Penganiayaan
- Pengadilan: PN RANTAU PRAPAT
- Pasal utama: Pasal 351 ayat (1) KUHP
- Jumlah dokumen: 30 PDF

Output utama dari notebook ini:
- Folder `data/raw/` berisi 30 file PDF
- Folder `data/text/` berisi hasil ekstraksi teks dari setiap PDF
- File `data/processed/cleaning_log.csv` berisi log hasil ekstraksi dan validasi teks


## 1. Instalasi Library

Cell ini digunakan untuk memasang library yang dibutuhkan. Library `pymupdf` digunakan untuk membaca teks dari file PDF, sedangkan `pandas` digunakan untuk membuat log hasil preprocessing.


In [ ]:
!pip -q install pymupdf pandas tqdm

## 2. Import Library

Cell ini memanggil library yang akan digunakan selama proses pembangunan case base, mulai dari pengelolaan file, ekstraksi PDF, pembersihan teks, hingga penyimpanan log.


In [ ]:
import os
import re
import shutil
import zipfile
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm

print("Library berhasil dimuat.")

## 3. Membuat Struktur Folder Project

Cell ini membuat struktur folder sesuai kebutuhan tugas. Folder `data/raw/` digunakan untuk menyimpan PDF asli, sedangkan `data/text/` digunakan untuk menyimpan hasil ekstraksi teks bersih.


In [ ]:
# Lokasi project di Google Colab
BASE_DIR = Path("/content/cbr-putusan-penganiayaan")

# Struktur folder utama
RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOG_DIR = BASE_DIR / "logs"

# Membuat folder jika belum ada
for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Struktur folder berhasil dibuat:")
print(BASE_DIR)

## 4. Upload ZIP Berisi 30 PDF

Cell ini digunakan untuk mengunggah file ZIP yang berisi 30 PDF putusan. Gunakan file ZIP yang sudah dirapikan dengan nama `case_001.pdf` sampai `case_030.pdf`.

Jika file ZIP sudah ada di folder `/content/`, cell ini akan langsung membacanya. Jika belum ada, Colab akan meminta upload file.


In [ ]:
# Nama ZIP yang disarankan
EXPECTED_ZIP_NAME = "valid_case_001_030.zip"

zip_path = Path("/content") / EXPECTED_ZIP_NAME

if not zip_path.exists():
    try:
        from google.colab import files
        print("Silakan upload file ZIP berisi 30 PDF.")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        zip_path = Path("/content") / list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            "File ZIP belum tersedia. Upload file ZIP ke Colab terlebih dahulu."
        ) from e

print(f"File ZIP digunakan: {zip_path}")

## 5. Ekstraksi ZIP ke Folder `data/raw/`

Cell ini mengekstrak file ZIP, mengambil seluruh file PDF, lalu menyalinnya ke folder `data/raw/`. File selain PDF, seperti mapping CSV, akan diabaikan.


In [ ]:
# Bersihkan folder raw agar tidak tercampur dengan percobaan sebelumnya
for old_file in RAW_DIR.glob("*.pdf"):
    old_file.unlink()

TEMP_EXTRACT_DIR = BASE_DIR / "temp_extract"

if TEMP_EXTRACT_DIR.exists():
    shutil.rmtree(TEMP_EXTRACT_DIR)
TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Ekstrak ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(TEMP_EXTRACT_DIR)

# Ambil semua PDF dari hasil ekstraksi
pdf_files = sorted(TEMP_EXTRACT_DIR.rglob("*.pdf"))

if len(pdf_files) == 0:
    raise FileNotFoundError("Tidak ditemukan file PDF di dalam ZIP.")

# Fungsi sorting berdasarkan angka pada nama case_001, case_002, dst.
def case_sort_key(path):
    match = re.search(r"case_(\d+)", path.name.lower())
    if match:
        return int(match.group(1))
    return path.name.lower()

pdf_files = sorted(pdf_files, key=case_sort_key)

# Salin ke data/raw/
for i, pdf_file in enumerate(pdf_files, start=1):
    new_name = f"case_{i:03d}.pdf"
    shutil.copy2(pdf_file, RAW_DIR / new_name)

raw_pdfs = sorted(RAW_DIR.glob("*.pdf"))

print(f"Jumlah PDF berhasil disalin ke data/raw/: {len(raw_pdfs)}")
for pdf in raw_pdfs[:5]:
    print("-", pdf.name)

if len(raw_pdfs) != 30:
    print("PERINGATAN: Jumlah PDF bukan 30. Periksa kembali file ZIP yang digunakan.")

## 6. Fungsi Ekstraksi dan Pembersihan Teks

Cell ini berisi fungsi untuk membaca teks dari PDF, menghapus watermark, menghapus nomor halaman, menormalkan spasi, dan membersihkan teks agar siap dipakai pada tahap berikutnya.


In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Mengekstrak teks dari file PDF menggunakan PyMuPDF.
    """
    doc = fitz.open(pdf_path)
    pages_text = []

    for page in doc:
        text = page.get_text("text")
        pages_text.append(text)

    doc.close()
    return "\n".join(pages_text)


def clean_legal_text(text):
    """
    Membersihkan teks putusan dari watermark, disclaimer, nomor halaman,
    spasi berlebih, dan karakter yang tidak diperlukan.
    """
    if not isinstance(text, str):
        return ""

    # Normalisasi newline
    text = text.replace("\r", "\n")

    # Hapus watermark/teks berulang yang sering muncul
    repeated_patterns = [
        r"Mahkamah Agung Republik Indonesia",
        r"Direktori Putusan Mahkamah Agung Republik Indonesia",
        r"putusan\.mahkamahagung\.go\.id",
        r"Kepaniteraan Mahkamah Agung Republik Indonesia.*?Halaman \d+",
        r"Disclaimer.*?Halaman \d+",
        r"Email\s*:\s*kepaniteraan@mahkamahagung\.go\.id.*?Halaman \d+",
    ]

    for pattern in repeated_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.DOTALL)

    # Hapus pola halaman
    text = re.sub(r"Halaman\s+\d+\s+dari\s+\d+\s+Putusan\s+Nomor\s+[^\\n]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Halaman\s+\d+", " ", text, flags=re.IGNORECASE)

    # Perbaiki pemenggalan kata akibat PDF
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Normalisasi spasi dan newline
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" +\n", "\n", text)
    text = re.sub(r"\n +", "\n", text)

    return text.strip()


def count_words(text):
    """
    Menghitung jumlah kata pada teks.
    """
    return len(re.findall(r"\b\w+\b", text))


def validate_text(text, min_words=500):
    """
    Validasi sederhana untuk memastikan hasil ekstraksi teks cukup layak.
    """
    word_count = count_words(text)

    if word_count >= min_words:
        return "valid"
    return "perlu_cek"


## 7. Konversi PDF ke Teks Bersih

Cell ini membaca seluruh PDF di folder `data/raw/`, mengekstrak isi teksnya, membersihkan hasil ekstraksi, lalu menyimpan file `.txt` ke folder `data/text/`.


In [ ]:
# Bersihkan folder text agar hasil baru tidak tercampur
for old_file in TEXT_DIR.glob("*.txt"):
    old_file.unlink()

cleaning_logs = []

for pdf_path in tqdm(sorted(RAW_DIR.glob("*.pdf")), desc="Memproses PDF"):
    case_id = pdf_path.stem
    txt_path = TEXT_DIR / f"{case_id}.txt"

    try:
        raw_text = extract_text_from_pdf(pdf_path)
        clean_text = clean_legal_text(raw_text)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(clean_text)

        cleaning_logs.append({
            "case_id": case_id,
            "pdf_file": pdf_path.name,
            "txt_file": txt_path.name,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean_text),
            "word_count": count_words(clean_text),
            "status": validate_text(clean_text),
            "error": ""
        })

    except Exception as e:
        cleaning_logs.append({
            "case_id": case_id,
            "pdf_file": pdf_path.name,
            "txt_file": "",
            "raw_char_count": 0,
            "clean_char_count": 0,
            "word_count": 0,
            "status": "error",
            "error": str(e)
        })

cleaning_log_df = pd.DataFrame(cleaning_logs)

log_path = PROCESSED_DIR / "cleaning_log.csv"
cleaning_log_df.to_csv(log_path, index=False)

print("Proses ekstraksi dan cleaning selesai.")
print(f"Log disimpan di: {log_path}")

## 8. Menampilkan Log Hasil Cleaning

Cell ini menampilkan ringkasan hasil ekstraksi teks. Dokumen dengan status `valid` berarti teksnya cukup panjang dan layak digunakan. Dokumen dengan status `perlu_cek` perlu diperiksa manual.


In [ ]:
cleaning_log_df

In [ ]:
print("Ringkasan status:")
print(cleaning_log_df["status"].value_counts())

print("\nJumlah file TXT yang berhasil dibuat:", len(list(TEXT_DIR.glob("*.txt"))))
print("Jumlah file PDF pada data/raw/:", len(list(RAW_DIR.glob("*.pdf"))))

## 9. Preview Teks Salah Satu Case

Cell ini menampilkan sebagian teks dari `case_001.txt` untuk memastikan bahwa teks hasil ekstraksi sudah terbaca dan dapat digunakan.


In [ ]:
sample_txt = TEXT_DIR / "case_001.txt"

if sample_txt.exists():
    with open(sample_txt, "r", encoding="utf-8") as f:
        sample_text = f.read()

    print(sample_text[:3000])
else:
    print("File case_001.txt belum ditemukan.")

## 10. Simpan Ringkasan Output Tahap 1

Cell ini membuat file ringkasan tahap 1 yang berisi jumlah PDF, jumlah file teks, dan lokasi output. File ini berguna sebagai bukti bahwa tahap build case base sudah berhasil dijalankan.


In [ ]:
summary = {
    "jumlah_pdf_raw": len(list(RAW_DIR.glob("*.pdf"))),
    "jumlah_txt": len(list(TEXT_DIR.glob("*.txt"))),
    "folder_pdf": str(RAW_DIR),
    "folder_text": str(TEXT_DIR),
    "cleaning_log": str(log_path),
}

summary_df = pd.DataFrame([summary])
summary_path = PROCESSED_DIR / "stage_01_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary_df

## 11. Membuat ZIP Output Tahap 1

Cell ini membuat file ZIP berisi folder project hasil tahap 1. ZIP ini dapat diunduh dari Google Colab atau dipakai untuk melanjutkan tahap berikutnya.


In [ ]:
output_zip = Path("/content/stage_01_build_case_base_output.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip).replace(".zip", ""),
    format="zip",
    root_dir=BASE_DIR
)

print(f"ZIP output berhasil dibuat: {output_zip}")

## Kesimpulan Tahap 1

Pada tahap ini, 30 dokumen putusan PDF telah disimpan sebagai data mentah, diekstrak menjadi teks, dibersihkan, dan divalidasi secara sederhana. Hasil dari tahap ini akan digunakan pada tahap berikutnya, yaitu `02_case_representation.ipynb` untuk mengambil metadata penting seperti nomor perkara, terdakwa, pasal, amar putusan, dan ringkasan fakta.
